# OpenAI 플랫폼 API를 활용한 GPT Fine-Tuning

# 0. API Key 로드 

In [13]:
from dotenv import load_dotenv

load_dotenv()

True

# 1. 데이터셋 구축 

파인튜닝을 위한 학습용 데이터셋 구축 

- **적절한 examples 수**
    - 최소: 10개의 examples 요구됨
    - 권장: 50~100개의 examples
        - 해당 개수로 성능향상이 관찰되지만, 필요한 정확한 수는 사용 사례에 따라 크게 달라질 수 있음
    - 전략: 50개 테스트 -> 성능 확인 -> 점진적 추가 
        - 우선, 잘 만들어진 50개의 데모 예시로 파인튜닝 후 결과를 우선 평가해보기 
        - 만약 50개의 양질의 examples로 성능이 개선된다면, 더 많은 예시를 추가하여 추가적인 성능향상이 있는지 확인 
        - 반면, 50개의 예시로도 전혀 성능차이가 없다면, 데이터를 추가하기 전에 작업 자체나 프롬프트를 다시 고민해볼필요가 있음 

- **좋은 examples 조건**
    - 실제 어플리케이션에서 기대하는 프롬프트와 출력 형식에 최대한 가깝게 구성 (실제 사용 사례와 유사)
    - 구체적이고 명확한 질문-답변
    - 일관된 포맷과 톤 
    - 필요시 도메인 전문지식 반영
        - 과거 데이터, 전문가 작성 데이터, 로그된 데이터 등 다양하게 수집된 데이터 활용 

- **데이터 포맷팅**
    - 형식: JSONL(JSON Lines)
        - 학습 데이터 파일의 각 line마다 하나의 완전한 JSON 구조가 있어야 함
    - 구조: Chat completions
    - 개수: 최소 10 lines 이상

In [14]:
import json 

finance_data_path = "../data/fi_train_data.jsonl"

def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        dataset = [json.loads(line) for line in f]
    return dataset

fi_dataset = load_data(finance_data_path)
print(len(fi_dataset))
display(fi_dataset[0])

10


{'messages': [{'role': 'system',
   'content': '당신은 국민은행·신한은행·하나은행의 전세자금대출 상품설명서를 기반으로 고객 맞춤 대출 진단 및 관리를 제공하는 AI 금융 상담사입니다. 모든 답변은 아래 포맷을 반드시 준수하세요.\n\n[진단결과] 고객 상황 핵심 요약 (1~2문장)\n[적합상품] 추천 은행 및 상품명\n[핵심조건] 금리·한도·보증기관 등 주요 조건\n[주의사항] 중도상환해약금·연장절차·연체불이익 등 유의사항\n[다음단계] 고객이 바로 해야 할 행동 (1~3가지)'},
  {'role': 'user', 'content': '안녕하세요. 신혼부부인데 하나은행 전세자금대출을 알아보고 있어요.'},
  {'role': 'assistant',
   'content': '[진단결과] 신혼부부 무주택 세대로 전세자금대출 자격 요건 검토가 필요합니다.\n[적합상품] 하나은행 전세자금대출 (전세금안심대출 또는 우량전세자금대출)\n[핵심조건] 임차보증금반환채권에 질권 설정 방식으로 담보 취급, 금리는 COFIX 또는 금융채 기준 변동금리·혼합금리·고정금리 중 선택 가능\n[주의사항] ① 대출 만기 전 상환 시 3년 이내 중도상환해약금 발생 ② 전세계약 연장 시 대출이 자동 연장되지 않으므로 만기 1개월 전 은행 신청 필수 ③ 임대차보증금 반환 시 임차인이 직접 수령 불가\n[다음단계] 1) 소득 증빙·전세계약서 사전 준비 2) 하나은행 고객센터(1599-2222) 사전심사 신청 3) 임대인 협조(질권설정 통지·승낙) 사전 확인'},
  {'role': 'user', 'content': '연소득이 합산 6,000만원이고 전세금은 3억원이에요. 한도가 얼마나 될까요?'},
  {'role': 'assistant',
   'content': '[진단결과] 합산 연소득 6,000만원, 전세금 3억원으로 한도는 DSR·보증기관 기준 복합 산출됩니다.\n[적합상품] 하나은행 전세자금대출 (주택도시보증공사·서울보증보험 보증서 연계)\n[

In [15]:
for i in range(len(fi_dataset[0]["messages"])):
    print(fi_dataset[0]["messages"][i]["content"])
    print("-"*70)

당신은 국민은행·신한은행·하나은행의 전세자금대출 상품설명서를 기반으로 고객 맞춤 대출 진단 및 관리를 제공하는 AI 금융 상담사입니다. 모든 답변은 아래 포맷을 반드시 준수하세요.

[진단결과] 고객 상황 핵심 요약 (1~2문장)
[적합상품] 추천 은행 및 상품명
[핵심조건] 금리·한도·보증기관 등 주요 조건
[주의사항] 중도상환해약금·연장절차·연체불이익 등 유의사항
[다음단계] 고객이 바로 해야 할 행동 (1~3가지)
----------------------------------------------------------------------
안녕하세요. 신혼부부인데 하나은행 전세자금대출을 알아보고 있어요.
----------------------------------------------------------------------
[진단결과] 신혼부부 무주택 세대로 전세자금대출 자격 요건 검토가 필요합니다.
[적합상품] 하나은행 전세자금대출 (전세금안심대출 또는 우량전세자금대출)
[핵심조건] 임차보증금반환채권에 질권 설정 방식으로 담보 취급, 금리는 COFIX 또는 금융채 기준 변동금리·혼합금리·고정금리 중 선택 가능
[주의사항] ① 대출 만기 전 상환 시 3년 이내 중도상환해약금 발생 ② 전세계약 연장 시 대출이 자동 연장되지 않으므로 만기 1개월 전 은행 신청 필수 ③ 임대차보증금 반환 시 임차인이 직접 수령 불가
[다음단계] 1) 소득 증빙·전세계약서 사전 준비 2) 하나은행 고객센터(1599-2222) 사전심사 신청 3) 임대인 협조(질권설정 통지·승낙) 사전 확인
----------------------------------------------------------------------
연소득이 합산 6,000만원이고 전세금은 3억원이에요. 한도가 얼마나 될까요?
----------------------------------------------------------------------
[진단결과] 합산 연소득 6,000만원, 전세금 3억원으로

# 2. 데이터셋 검증 

- Fine-Tuning을 진행하기에 적절한 데이터셋인지 검증 해보자 
    - Fine-Tuning은 시간과 비용이 많이 들 수 있기 때문에 사전에 적합한 데이터 형태인지 검증하고 진행하는 게 안정적 
    - OpenAI의 Fine-Tuning API 사용 시, 데이터의 구조는 JSONL로 구성돼야 함 
    - JSONL (JSON Lines): 일반 JSON이 전체 데이터를 하나의 객체로 보는것에 비해, JSONL은 한 줄당 JSON객체가 존재하는 형태로 한 줄씩 또는 청크단위로 읽고 쓰는 것에 효율적이며, 대규모의 데이터셋을 구성하는데 적합함 

In [16]:
import json 
import tiktoken 
import numpy as np 
from collections import defaultdict 

## 2.1 데이터셋 포맷 검증 

- 올바른 형식(포맷)으로 작성되었는지 검증 

In [17]:
# defaultdict 사용하지 않을 경우 
temp = {}
# temp["test"] = 0
temp["test"] += 1
temp

KeyError: 'test'

In [18]:
# defaultdict를 활용할 경우 
temp = defaultdict(int)
temp["test"] += 1 
temp 

defaultdict(int, {'test': 1})

In [20]:
def validate_dataset(dataset, domain_name):
    """데이터셋 포맷 검증"""
    print("="*60)
    print(f"[ {domain_name} 데이터셋 검증 ]")

    # 데이터셋 포맷 검증 확인용 dict생성 
    format_errors = defaultdict(int)

    for ex in dataset:
        # 데이터 타입 검증: ex가 dict가 아닐 경우 카운팅 
        if not isinstance(ex, dict):
            format_errors["data_type"] += 1
            continue

        # 한 개의 line에 key가 존재하는지 확인: 없으면 카운팅
        messages = ex.get("messages", None) # messages가 존재하면 해당 value반환, 없으면 None반환
        if not messages: # messages에 데이터가 존재하지 않을 경우 (None일 경우)
            format_errors["missing_messages_list"] += 1 
            continue

        # role 검증 
        for message in messages:
            # role & content 존재 여부 검증 
            if ("role" not in message) or ("content" not in message):
                format_errors["message_missing_key"] += 1

            # 각 message에 쓰이지 않는 key 존재 여부 검증 
            if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
                # any(): iterable객체 내에 하나라도 True인 값이 있으면 전체 결과를 True로 반환 
                format_errors["message_unrecognized_key"] += 1
            
            # content & function_call 존재 여부 검증 
            content = message.get("content", None)
            function_call = message.get("function_call", None)
            if (not content and not function_call) or not isinstance(content, str):
                format_errors["missing_content"] += 1

        # role 중에 assistant 존재 여부 검증 
        if not any(message.get("role", None) == "assistant" for message in messages):
            format_errors["example_missing_assistant_message"] += 1

    # 에러 존재 여부 최종 확인 
    if format_errors:
        print(f"Found errors in {domain_name}") 
        for k, v in format_errors.items():
            print(f"* {k}: {v}")
    else:
        print(f"* No errors found in {domain_name}")

# 데이터셋 포맷 검증 진행 
validate_dataset(fi_dataset, "금융 데이터셋")

[ 금융 데이터셋 데이터셋 검증 ]
* No errors found in 금융 데이터셋


## 2.2 메시지별 토큰 수 확인용 함수 정의 

In [21]:
# not exact!
# simplified from https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb

# 인코딩 객체 정의 (message별 value값 인코딩 위함)
# tiktoken: OpenAI 모델들이 사용하는 토큰화 및 인코딩 지원 모듈
# .get_encoding(): 토큰화 및 인코딩 모델 검색 
# "o200k_base": gpt-4o 및 gpt-4o-mini가 사용하는 모델명(그 외 모델들은 "cl100k_base" 모델명을 사용)

encoding = tiktoken.encoding_for_model("gpt-4.1-nano-2025-04-14") # 해당 모델의 인코더 선택

# 메시지당 토큰 수 계산: 메시지당 3토큰, role당 1토큰 
def num_tokens_from_messages(messages, token_per_message=3):
    num_tokens = 0
    for message in messages:
        num_tokens += token_per_message # messages list에 message 개수 계산
        for _, value in message.items():
            num_tokens += len(encoding.encode(value)) # 각 message의 value 인코딩 (role, content에 해당되는 value 인코딩)

    # 모델 내부에서는 각 role, content가 바뀔 때마다 이를 구분지어주는 <start>, <end> 토큰이 붙어서 토큰이 계산됨
    # 해당 고정 토큰을 계산하기 위해 추가로 3토큰을 더해줌 
    # 아래 셀에서 예시를 확인해주세요
    num_tokens += 3 
    return num_tokens

# 메시지당 assistant에 쓰인 토큰 수 계산 
def num_assistant_tokens_messages(messages):
    num_tokens = 0
    for message in messages:
        if message["role"] == "assistant":
            num_tokens += len(encoding.encode(message["content"]))
    return num_tokens

# 각 메시지별로 쓰인 토큰 개수 통계치 출력 함수 
def print_distribution(values, name):
    print(f"\n#### Distribution of {name}")
    print(f"min / max: {min(values)}, {max(values)}")
    print(f"mean / median: {np.mean(values)}, {np.median(values)}")

In [22]:
# 인코딩 예제
encoding = tiktoken.encoding_for_model("gpt-4.1-nano-2025-04-14")
print(encoding.encode("hello, my name is heeyoon!"))

# hello, my, name, is, heeyoon, !

[24912, 11, 922, 1308, 382, 501, 806, 4508, 0]


In [ ]:
# 우리가 보는 메시지 
"""
user: 안녕하세요
assistant: 반갑습니다
"""

# 모델 내부
"""
<|start|>system
...
<|end|>
<|start|>user
안녕하세요
<|end|>
<|start|>assistant
반갑습니다
<|end|>
"""



## 2.3 함수 적용: 데이터 경고 및 토큰 수 분석 

In [23]:
def analysis_tokens(dataset, domain_name):
    """데이터셋 토큰 수 분석"""
    print("="*60)
    print(f"[ {domain_name} 메시지별 토큰 수 분석 ]")

    n_missing_system = 0            # system 누락 수
    n_missing_user = 0              # user 누락 수 
    n_messages = []                 # 예제당 메시지 수: 각 대화에서 메시지 수의 분포를 요약하여 대화의 복잡성 파악
    convo_lens = []                 # 예제당 총 토큰 수: 각 대화의 총 토큰수를 계산하고 분포 파악
    assistant_message_lens = []     # assistant 메시지 토큰 수: 대화당 ai가 생성한 토큰수를 계산하고 분포 파악(ai의 장황함을 파악하는 데 유용)

    for ex in dataset:
        messages = ex["messages"]
        if not any(message["role"] == "system" for message in messages):
            n_missing_system += 1
        if not any(message["role"] == "user" for message in messages):
            n_missing_user += 1

        n_messages.append(len(messages))
        convo_lens.append(num_tokens_from_messages(messages))
        assistant_message_lens.append(num_assistant_tokens_messages(messages))

    # 토큰 통계 및 분포 확인 
    print("누락된 system 수:", n_missing_system)
    print("누락된 user 수:", n_missing_user)
    print_distribution(n_messages, "예제당 메시지 수")
    print_distribution(convo_lens, "예제당 총 토큰 수")
    print_distribution(assistant_message_lens, "예제당 assistant 토큰 수")

    # 경구 
    n_too_long = sum(l > 1047576 for l in convo_lens)
    print(f"\n{n_too_long}개의 예제수는 1,047,576개의 토큰 제한을 넘어서며, fine-tuning 중 잘릴 수 있습니다.")

    # 추후 비용 계산을 위해 예제당 총 토큰 수 반환 
    return convo_lens

In [24]:
fi_convo = analysis_tokens(fi_dataset, "금융 데이터셋")

[ 금융 데이터셋 메시지별 토큰 수 분석 ]
누락된 system 수: 0
누락된 user 수: 0

#### Distribution of 예제당 메시지 수
min / max: 5, 7
mean / median: 5.6, 5.0

#### Distribution of 예제당 총 토큰 수
min / max: 700, 1010
mean / median: 814.4, 757.0

#### Distribution of 예제당 assistant 토큰 수
min / max: 482, 762
mean / median: 587.2, 537.0

0개의 예제수는 1,047,576개의 토큰 제한을 넘어서며, fine-tuning 중 잘릴 수 있습니다.


## 2.4 비용 추정

- 파인튜닝에 사용될 총 토큰 수 추정 -> 대략적인 비용 계산해보기
- 토큰 수가 증가할수록 파인튜닝 작업의 시간 소요 시간도 길어짐에 주의 
- 데이터의 양과 원하는 epochs수에 따라 최저기준과 최대기준을 비교하여 적정 eopchs를 구하고 적용하는 방식 

In [25]:
def calculate_cost(dataset, convo_lens, domain_name):
    """파인튜닝 비용 추정"""
    MAX_TOKENS_PER_EXAMPLE = 1047576 # 모델에서 허용하는 대화당(예제당) 최대 토큰 수 

    # fine-tuning 시 적정 epochs 지정 
    TARGET_EPOCHS = 3           # 기본 base 에포크 수 
    MIN_DEFAULT_EPOCHS = 1      # OpenAI가 제시하는 fine-tuning에 필요한 최소 epochs 수
    MAX_DEFAULT_EPOCHS = 25     # OpenAI가 제시하는 fine-tuning에 필요한 최대 epochs 수 (해당 횟수를 초과하면 과대적합에 빠질 가능성이 높음)

    MIN_TARGET_EXAMPLES = 100   # OpenAI가 제시하는 fine-tuning에 필요한 최소 예제 수
    MAX_TARGET_EXAMPLES = 25000 # OpenAI가 제시하는 fine-tuning에 필요한 최대 예제 수 (비용 및 시간 효율을 위해 상한 설정)

    n_epochs = TARGET_EPOCHS        # 입력되는 타겟 에포크에 따라 실제로 적용될 에폭(n_epochs) 설정
    n_train_examples = len(dataset) # 예제 개수 

    # 학습되는 총 예제 수(예제개수x에폭)가 OpenAI에서 제시하는 최소 예제 수 보다 적을 경우 
    if n_train_examples * TARGET_EPOCHS < MIN_TARGET_EXAMPLES:
        # 학습되는 예제수가 너무 적을 경우 최소 epochs와 비교하여 적정한 epochs 재설정 -> epochs로 학습량 조절
        n_epochs = min(MAX_DEFAULT_EPOCHS, MIN_TARGET_EXAMPLES // n_train_examples) # 두 인자 중 더 작은 값 반환
    
    # 학습되는 총 예제 수(예제개수x에폭)가 OpenAI에서 제시한 최소 예제 수보다 클 경우 
    elif n_train_examples * TARGET_EPOCHS > MAX_TARGET_EXAMPLES:
        # 학습되는 예제수가 너무 많을 경우 최소 epochs와 비교하여 적정한 epochs 재설정
        n_epochs = max(MIN_DEFAULT_EPOCHS, MAX_TARGET_EXAMPLES // n_train_examples)

    # convo_lens: 예제당 총 토큰수 모아놓은 list 
    # MAX_TOKENS_PER_EXAMPLE와 비교하는 이유: 예제별로 쓰이는 토큰 수가 context window 수를 넘길 수 없기 때문
    n_billing_tokens = sum(min(MAX_TOKENS_PER_EXAMPLE, length) for length in convo_lens)

    print(f"\n📊 {domain_name} 비용 추정")
    print("="*60)
    print(f"총 예제: {n_train_examples}개")
    print(f"1 epoch 토큰: {n_billing_tokens:,}")
    print(f"권장 epochs: {n_epochs}")
    print(f"총 {n_epochs * n_billing_tokens}개의 토큰에 대해 요금이 청구됩니다.")

calculate_cost(fi_dataset, fi_convo, "금융 데이터셋")


📊 금융 데이터셋 비용 추정
총 예제: 10개
1 epoch 토큰: 8,144
권장 epochs: 10
총 81440개의 토큰에 대해 요금이 청구됩니다.


# 3. Fine-Tuning 파일 객체 생성 

- 데이터셋을 OpenAI 서버에 전달하여 fine-tuning에 쓰이도록, 또는 진행 과정을 추적할 수 있도록 객체 생성 

In [26]:
from openai import OpenAI 

client = OpenAI()

def upload_file_obj(path, domain_name):
    finetuning_file = client.files.create(
        file=open(path, "rb"),
        purpose="fine-tune"
    )
    print(f"✅ {domain_name}파일 업로드 완료: {finetuning_file.id}")
    return finetuning_file.id

# OpenAI 플랫폼의 Dashboard의 Storage에서 파일이 업로드 됐는지 확인하기 
fi_obj_id = upload_file_obj(finance_data_path, "금융 데이터셋")

✅ 금융 데이터셋파일 업로드 완료: file-4ZsYz3mPtM6RLcRTEdy1mX


# 4. Fine-Tuning 시작 

## 4.1 작업 객체 생성 

- 파인튜닝 작업 생성: https://platform.openai.com/docs/api-reference/fine-tuning
    - 파인튜닝용 데이터셋을 업로드한 후, 제공한 데이터셋을 사용하여 기본 gpt모델을 사용자 정의하는 파인튜닝 작업 생성하기 
    - 파인튜닝 작업을 생성할 때는 다음을 반드시 지정해야 함 
        - 기본모델(model): 파인튜닝에 사용할 OpenAI모델 ID나, 이전에 파인튜닝된 모델의 ID (지원되는 모델: https://platform.openai.com/docs/guides/model-optimization#fine-tune-a-model)
        - 훈련 파일(training_file): 이전 단계에서 업로드한 파일 
        - 파인튜닝 팡법(method): 모델을 어떻게 사용자 정의할건지 지정(default: 감독 학습 기반 파인튜닝, https://platform.openai.com/docs/guides/model-optimization#fine-tuning-methods)

In [27]:
# 아래코드 실행 후 OpenAI Dashboard에서 진행상태 확인 
# epochs는 "auto"로 설정되어 학습됨 -> 조절하고 싶은 경우 hyperparameters인자에서 n_epochs key에 지정 가능

# 해당 셀은 실행하자마자 파인튜닝 작업이 시작되기 때문에 주의하기 
finance_job = client.fine_tuning.jobs.create(
    model="gpt-4.1-nano-2025-04-14",
    training_file=fi_obj_id,
    suffix="finance-loan-expert-multiturn" # 내가 만든 파인튜닝 모델에 붙이는 별명(식별자)
) 

print("✅ 금융 모델 파인튜닝 시작")
print(f"job ID: {finance_job.id}")
print(f"status: {finance_job.status}")
finance_job # 작업 상태에 대한 정보가 포함되어있음 

✅ 금융 모델 파인튜닝 시작
job ID: ftjob-fftMn6SutKfYA3xWdgC9mKef
status: validating_files


FineTuningJob(id='ftjob-fftMn6SutKfYA3xWdgC9mKef', created_at=1772084004, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-y2IT7FAyjAIVT6tBKowGn7xn', result_files=[], seed=2101723209, status='validating_files', trained_tokens=None, training_file='file-4ZsYz3mPtM6RLcRTEdy1mX', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='finance-loan-expert-multiturn', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

## 4.2 작업 진행 현황 파악 

In [28]:
# 특정 작업 파인튜닝 진행 위한 id값 출력: 해당 아이디값을 통해 특정 작업의 진행상태를 확인할 수 있음 
finance_job.id 

'ftjob-fftMn6SutKfYA3xWdgC9mKef'

In [29]:
# 작업 id에 따른 세부사항 확인 (작업 id, 작업 완료된 모델명, 하이퍼파리미터, 현재 진행상태 등)

client.fine_tuning.jobs.retrieve(finance_job.id)

FineTuningJob(id='ftjob-fftMn6SutKfYA3xWdgC9mKef', created_at=1772084004, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-y2IT7FAyjAIVT6tBKowGn7xn', result_files=[], seed=2101723209, status='validating_files', trained_tokens=None, training_file='file-4ZsYz3mPtM6RLcRTEdy1mX', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix='finance-loan-expert-multiturn', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

In [30]:
import time 

# 30초마다 진행상태 확인하기 
while True:
    finance_job_info = client.fine_tuning.jobs.retrieve(finance_job.id)
    print(f"진행상태: {finance_job_info.status}")

    if finance_job_info.status == "succeeded":
        print("fine-tuning 완료")
        break

    if finance_job_info.status == "failed":
        print("fine-tuning 실패")
        break

    time.sleep(30)

진행상태: validating_files
진행상태: validating_files
진행상태: validating_files
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: running
진행상태: succeeded
fine-tuning 완료


In [31]:
# 파인튜닝에 걸린 시간 확인
from datetime import datetime, UTC

# fromtimestamp: unix시간을 UTC(세계표준시간)으로 변환(한국 시간과 9시간 차이)
# unix 시간: 유닉스 운영체제에서 사용하는 컴퓨터 시간 표시 방법으로, 1970년 1월 1일을 기준으로 현재까지 몇초가 지났
start_time = datetime.fromtimestamp(finance_job_info.created_at, UTC) # created_at: 작업 시작
finish_time = datetime.fromtimestamp(finance_job_info.finished_at, UTC) # finished_at: 작업

print("작업 시작 시간:", start_time)
print("작업 완료 시간:", finish_time)
print("작업에 소요된 시간:", finish_time - start_time)

작업 시작 시간: 2026-02-26 05:33:24+00:00
작업 완료 시간: 2026-02-26 05:39:15+00:00
작업에 소요된 시간: 0:05:51


## 4.3 진행중인 작업 취소

In [ ]:
# client.fine_tuning.jobs.cancel(finance_job.id)

# 작업 시작 후에 잘못 설정된 것을 알게된 경우 중간에 취소하게 되면 불필요한 비용이나 시간낭비를 피할 수 있음
# OpenAI Dashboard에서 cancel 버튼을 통해 취소 가능 

## 4.4 완료된 모델 삭제 

In [ ]:
# client.models.delete(finance_job.fine_tuned_model)

# 사용하지 않는 모델이 많으면 id값을 잘못 입력할 경우 이전 버전 모델로 금액이 빠져나갈 수 있기 때문에 
# 정리차원에서 불필요한 모델은 미리 삭제해두는 게 좋음 

## 4.5 작업 목록 출력

- 진행했던 전체 작업 목록 출력 

In [32]:
jobs = client.fine_tuning.jobs.list(limit=3).data # limit: 출력 개수 제한

print(len(jobs))
jobs[0]  # 0번째가 가장 최근 작업물 

1


FineTuningJob(id='ftjob-fftMn6SutKfYA3xWdgC9mKef', created_at=1772084004, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:sesac:finance-loan-expert-multiturn:DDONggCe', finished_at=1772084355, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=10), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-y2IT7FAyjAIVT6tBKowGn7xn', result_files=['file-MtUT81uxb4YyqF6kjdRAjj'], seed=2101723209, status='succeeded', trained_tokens=81240, training_file='file-4ZsYz3mPtM6RLcRTEdy1mX', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=10))), user_provided_suffix='finance-loan-expert-multiturn', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

# 5. Fine-Tuning 완료 후 질의 응답 진행 

강의록 참고후 수정

In [36]:
FINETUNED_MODEL = client.fine_tuning.jobs.list(limit=3).data
FINETUNED_MODEL 

[FineTuningJob(id='ftjob-fftMn6SutKfYA3xWdgC9mKef', created_at=1772084004, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:sesac:finance-loan-expert-multiturn:DDONggCe', finished_at=1772084355, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=10), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-y2IT7FAyjAIVT6tBKowGn7xn', result_files=['file-MtUT81uxb4YyqF6kjdRAjj'], seed=2101723209, status='succeeded', trained_tokens=81240, training_file='file-4ZsYz3mPtM6RLcRTEdy1mX', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=10))), user_provided_suffix='finance-loan-expert-multiturn', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None

In [37]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 파인튜닝 완료된 모델 ID 입력
FINETUNED_MODEL = client.fine_tuning.jobs.list(limit=3).data[0].fine_tuned_model

# 프롬프트 작성

system = """
    당신은 국민은행·신한은행·하나은행의 전세자금대출 상품설명서를 기반으로 고객 맞춤 대출 진단 및 관리를 제공하는 AI 금융 상담사입니다. 모든 답변은 아래 포맷을 반드시 준수하세요.

[진단결과] 고객 상황 핵심 요약 (1~2문장)
[적합상품] 추천 은행 및 상품명
[핵심조건] 금리·한도·보증기관 등 주요 조건
[주의사항] 중도상환해약금·연장절차·연체불이익 등 유의사항
[다음단계] 고객이 바로 해야 할 행동 (1~3가지)\
"""

input_data =  "신한은행 전세자금대출에서 고정금리, 변동금리, 혼합금리 중 어떤 게 나은가요?"

template = ChatPromptTemplate([
    ("system", system),
    ("user", "{input_data}")
]) 

# model 정의
base_model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
finetuned_model = ChatOpenAI(model=FINETUNED_MODEL)

# 기본 모델 응답 생성
print("="*80)
print("[기본 모델 응답]")
print("="*80)
base_chain = template | base_model
base_response = chain.invoke(input_data)
print(base_response.content)

# 파인튜닝된 모델 응답 생성 
print("="*80)
print("[파인튜닝 모델 응답]")
print("="*80)
finetuned_chain = template | base_model
finetuned_response = chain.invoke(input_data)
print(finetuned_response.content)

[기본 모델 응답]


NameError: name 'chain' is not defined

# 6. 추가 Fine-Tuning 진행

## 6.1 결과가 좋다고 판단되지 않을 경우 

- 파인튜닝 후 결과가 좋다고 판단되지 않을 경우 대응 
    1. 데이터 확인 
        1) 데이터 추가: 가장 효과가 확실할 수 있음
        2) 데이터의 균형과 다양성 확인 
            - 너무 일관된 데이터 뿐만 아니라 원하는 응답에 대한 다양한 데이터를 생성 
            - 일반적으로, 적은 양의 고품질 데이터가 많은 양의 저품질 데이터 보다 효과적
        3) 기존 데이터의 문제점 조사
            - 원치 않는 질의-응답 방식의 대화 데이터가 포함되어 있는지 확인 (잘못된 패턴이 학습되었을 수 있음)
            - 응답에 필요한 모든 정보가 포함되어 있는지 확인 (할루시네이션 발생 가능성 존재)
            - 여러 사람들이 만든 데이터를 합친 경우 -> 일관성 확인해보기 (패턴을 찾기 힘든 데이터일 수 있음)
    2. 모델 확인
        1) 완료된 모델에 추가로 파인튜닝 진행: 완료된 모델명으로 추가 작업 가능
        2) 하이퍼파라미터 변경: epochs, learning_rate 등 변경해보기 

## 6.2 체크포인트(중간 저장)모델 성능 판단 

- chekpoint list api 
    - https://platform.openai.com/docs/api-reference/fine-tuning/list-checkpoints
    - checkpoint
        - 학습 중간에 자동 저장된 임시 모델 
        - 큰 사이즈의 모델을 fine-tuning시킬 시 수천~수만 개의 데이터로 여러 epochs 반복 학습할 경우 몇 시간~며칠이 소요될 수 있음
        - 이 경우 중간에 서버가 꺼지면 전부 날아가게되며 이를 방지하기 위해 중간 저장본을 자동으로 생성해줌 

- 성능평가지표
    - Loss: 시퀀스(토큰)에 대한 loss로, 토큰별 정답일 확률에 대한 오차를 계산 
        - 모델이 토큰을 입력받은 후 다음 토큰에 대한 확률 분포를 생성
        - y가 1(정답)일 때 y(hat)이 1에 가까워질수록 오차는 0이 됨 

    - Mean Token Accuracy: 평균 토큰 정확도
        - (올바르게 예측된 토큰 수 / i번째 예제의 전체 토큰 수)를 전체 데이터의 수로 평균낸 값 
강의공지에 메시지 보내기


In [33]:
# 특정 작업의 id 출력
print(finance_job.id)

ftjob-fftMn6SutKfYA3xWdgC9mKef


In [34]:
# 특정 작업의 checkponts 목록 확인: 파이썬이 아닌 터미널의 shell 명령어를 통해 확인 가능
# 특정 작업의 id를 전달하여 확인 가능 

!curl https://api.openai.com/v1/fine_tuning/jobs/ftjob-fftMn6SutKfYA3xWdgC9mKef/checkpoints \
  -H "Authorization: Bearer $OPENAI_API_KEY"

{
  "error": {
    "message": "Incorrect API key provided: $OPENAI_***_KEY. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   244  100   244    0     0    590      0 --:--:-- --:--:-- --:--:--   593
100   244  100   244    0     0    590      0 --:--:-- --:--:-- --:--:--   593


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 파인튜닝 완료된 모델 ID 입력
FINETUNED_MODEL = client.fine_tuning.jobs.list(limit=3).data[0].fine_tuned_model

# 프롬프트 작성

system = """
    당신은 국민은행·신한은행·하나은행의 전세자금대출 상품설명서를 기반으로 고객 맞춤 대출 진단 및 관리를 제공하는 AI 금융 상담사입니다. 모든 답변은 아래 포맷을 반드시 준수하세요.

[진단결과] 고객 상황 핵심 요약 (1~2문장)
[적합상품] 추천 은행 및 상품명
[핵심조건] 금리·한도·보증기관 등 주요 조건
[주의사항] 중도상환해약금·연장절차·연체불이익 등 유의사항
[다음단계] 고객이 바로 해야 할 행동 (1~3가지)\
"""

input_data =  "신한은행 전세자금대출에서 고정금리, 변동금리, 혼합금리 중 어떤 게 나은가요?"

template = ChatPromptTemplate([
    ("system", system),
    ("user", "{input_data}")
]) 

# model 정의
base_model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
finetuned_model = ChatOpenAI(model=FINETUNED_MODEL)


# 파인튜닝된 모델 응답 생성 
print("="*80)
print("[파인튜닝 모델 응답]")
print("="*80)
finetuned_chain = template | base_model
finetuned_response = chain.invoke(input_data)
print(finetuned_response.content)

## 6.3 추가 Fine-Tuning 진행

In [ ]:
# 1. 추가로 학습시킬 데이터 불러와서 OpenAI 서버에 전달

# train용 형태 확인
train_path = "../data/fi_train_data_add.jsonl"
with open(train_path, "r", encoding="utf-8") as f:
    train_add_dataset = [json.loads(line) for line in f]

# validation용 형태 확인
val_path = "../data/fi_val_data.jsonl"
with open(val_path, "r", encoding="utf-8")as f:
    val_dataset = [json.loads(line) for line in f]

print("train 데이터셋 개수:", len(train_add_dataset))
print("validation 데이터셋 개수:", len(val_dataset))
print()
print(train_add_dataset[0])
print()
print(val_dataset[0])

# 서버에 데이터 전달
train_add_file = client.files.create(
    file=open(train_path, "rb"),
    purpose="fine-tune"
)

train_add_file = client.files.create(
    file=open(val_path, "rb"),
    purpose="fine-tune"
)

train 데이터셋 개수: 10
validation 데이터셋 개수: 10

{'messages': [{'role': 'system', 'content': '당신은 국민은행·신한은행·하나은행의 전세자금대출 상품설명서를 기반으로 고객 맞춤 대출 진단 및 관리를 제공하는 AI 금융 상담사입니다. 모든 답변은 아래 포맷을 반드시 준수하세요.\n\n[진단결과] 고객 상황 핵심 요약 (1~2문장)\n[적합상품] 추천 은행 및 상품명\n[핵심조건] 금리·한도·보증기관 등 주요 조건\n[주의사항] 중도상환해약금·연장절차·연체불이익 등 유의사항\n[다음단계] 고객이 바로 해야 할 행동 (1~3가지)'}, {'role': 'user', 'content': '취업한 지 1년 된 직장인인데 처음 독립해서 전세 구하려고 해요.'}, {'role': 'assistant', 'content': '[진단결과] 취업 1년차 직장인으로 소득 증빙과 신용 이력 형성 여부가 핵심 심사 요소입니다.\n[적합상품] 국민은행·신한은행·하나은행 전세자금대출 (청년 우대 조건 포함 검토)\n[핵심조건] ① 소득 증빙: 근로소득원천징수영수증 또는 건강보험료 납부확인서 ② 재직 확인: 재직증명서 필수 ③ 신용점수: 최초 대출로 신용이력 짧아도 직장 재직·소득으로 심사 가능\n[주의사항] ① 재직 1년 미만 시 일부 상품 제한 가능 (은행별 상이) ② 신용카드 연체 없이 관리해온 이력이 유리\n[다음단계] 1) 재직증명서·소득 증빙 서류 준비 2) 주거래 은행 앱에서 사전심사 신청 3) 임대차계약 체결 전 한도 확인 후 전세 물건 탐색'}, {'role': 'user', 'content': '연봉 3,500만원인데 보증금 1억 5,000만원 전세는 가능할까요?'}, {'role': 'assistant', 'content': '[진단결과] 연소득 3,500만원, 기존 부채 없음 가정 시 1억 5,000만원 전세는 DSR 기준 가능성이 있습니다.\n[적합상품] 국민은행·신한은행·하나은행 전세자금대출\n[핵심조건]

In [ ]:
# 2. 추가 fine-tuning 진행
finetuning_job_add = client.fine_tuning.jobs.create(
    model="ft:gpt-4.1-nano-2025-04-14:sesac:finance-loan-expert-multiturn:DDONggCe"
    training_file=train_add_file.id,
    validation_file=val_file.id,
    hyperparameters={
        "n_epochs": 5,
        "learning_rate_multiplier": 0.08
    }
)
finetuning_job_add

In [ ]:
finetuning_job_add.id

In [ ]:
!curl https://api.openai.com/v1/fine_tuning/jobs/ftjob-fftMn6SutKfYA3xWdgC9mKef \
  -H "Authorization: Bearer $OPENAI_API_KEY" 

# 7. 결론 

- valid_loss가 줄어들고 train_loss와의 편차도 줄어든다면 좋은방향으로 학습됐음을 의미
- valid_loss가 줄어들지 않았거나 train_loss와의 편차가 커졌다면 과대적합의 위험이 있음 (실제 판단은 gpt의 응답을 사람이 확인해봐야 함)
- 현재는 간단히 10개 정도의 데이터로 학습 및 추가학습을 진행했으므로 큰 성능 향상을 기대하기는 어려우며, 실전에서는 최소 100개 이상의 데이터로 fine-tuning을 진행해야 효과를 볼 가능성이 높음 
- GPT 모델 fine-tuning에 필요한 데이터는 오픈소스 데이터가 극히 드물기 때문에 프로젝트 시 직접 만드는 경우가 대부분 
    - 현업에서도 프로젝트 시 협력기업들과 데이터를 생성하거나 새로 수집하는 것부터 프로젝트를 진행함